In [ ]:
!pip install unsloth
!pip install trl accelerate bitsandbytes
!pip install rouge-score nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594

In [ ]:
import re
import pandas as pd
from datasets import Dataset, DatasetDict

# ── Constants ─────────────────────────────────────────────────────────────────
DATASET_PATH  = "/kaggle/input/datasets/raseluddin/bengali-empathetic-conversations-corpus/BengaliEmpatheticConversationsCorpus .csv"
ANSWER_MARKER = "[/INST]\nCounselor:\n"   # shared split key — never change

# ── 1. Load & clean ───────────────────────────────────────────────────────────
df = pd.read_csv(DATASET_PATH)
df.columns = df.columns.str.strip()
print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst few rows:")
df.head()

Dataset shape: (38233, 4)

Columns: ['Topics', 'Question-Title', 'Questions', 'Answers']

First few rows:


,Topics,Question-Title,Questions,Answers
0,পারিবারিক দ্বন্দ্ব,মা ও স্ত্রীর মধ্যে মতানৈক্য বৃদ্ধি,আমার স্ত্রী এবং মায়ের মধ্যে টানটান মতবিরোধ চ...,"আপনি যা বর্ণনা করছেন তাকে মনোবিজ্ঞানীরা ""ত্রি..."
1,"পদার্থের অপব্যবহার, আসক্তি",আমি ধূমপানে আসক্ত। আমি কিভাবে থামাতে পারি?,"আমি বাচ্চা নেওয়ার পরিকল্পনা করছি, তাই আমাকে ...",হাই। আপনার শিশুর (এবং নিজের) জন্য যা স্বাস্থ্...
2,পারিবারিক দ্বন্দ্ব,আমার পরিবারের কাছ থেকে গোপন রাখা,"আমার মনের মধ্যে গোপন আছে, এবং আমি জানি না তাদ...",মনে হচ্ছে গোপন রাখা এখন আপনার জন্য একটি সমস্য...
3,"আচরণগত পরিবর্তন, সামাজিক সম্পর্ক",অধিকারী হওয়ার অন্তর্নিহিত কারণ,আমি আমার সম্পর্কের ক্ষেত্রে অত্যন্ত অধিকারসূচক...,হ্যালো। এটা দুর্দান্ত যে আপনি উপলব্ধি করতে সক...
4,দুশ্চিন্তা,আমি কি ওষুধ ছাড়া উদ্বেগ নিয়ন্ত্রণ করতে পারি?,কয়েক বছর আগে আমার মাথায় আঘাত লেগেছিল এবং আমা...,আপনি বলেননি কি বা কত ওষুধ আপনি চেষ্টা করেছেন।...


In [ ]:
print("\nDataset Info:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38233 entries, 0 to 38232
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Topics          38222 non-null  object
 1   Question-Title  37639 non-null  object
 2   Questions       38215 non-null  object
 3   Answers         38228 non-null  object
dtypes: object(4)
memory usage: 1.2+ MB
None

Missing values:
Topics             11
Question-Title    594
Questions          18
Answers             5
dtype: int64


In [ ]:
for i,row in df.sample(3).iterrows():
    print("Topic:", row["Topics"])
    print("Question:", row["Questions"])
    print("Answer:", row["Answers"])
    print("="*80)

Topic: গর্বিত
Question: আপনাকে অনেক ধন্যবাদ. আমি নিজেকে নিয়ে খুব গর্বিত এবং এটি আমাকে আরও বেশি অনুপ্রেরণা দেয়!
Answer: হ্যাঁ, আমি সারা জীবন আমার ওজনের সাথে লড়াই করেছি।
Topic: আতঙ্কিত
Question: হ্যাঁ, কিন্তু সে রিং করেনি। আমি হতবাক হয়ে গিয়েছিলাম কিন্তু দেখা যাচ্ছে যে আমি জানালাটি খোলা রেখেছি এবং এটি ছিল স্যুট।
Answer: ওহ, বাহ, আমি ভেবেছিলাম হয়তো কোন প্রাণী ঢুকেছে
Topic: সংবেদনশীল
Question: না, আমরা যেখানে থাকি তার মধ্যে দূরত্বের কারণে এটি কঠিন। তাই আমরা সেই মুহূর্তগুলোকে উপলব্ধি করার চেষ্টা করি।
Answer: আমি সম্পর্ক করতে পারি। আমি শেষবার আমার মাকে দেখেছিলাম প্রায় ৮ বছর আগে এবং আমার বাবাকে প্রায় ১০ বছর আগে। হয়তো আপনার তাদের কাছে যাওয়া উচিত।


In [ ]:
from huggingface_hub import login

# Replace 'your_token_here' with your actual Hugging Face access token
login(token='')


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.1-8B-Instruct")

df["q_tokens"] = df["Questions"].apply(lambda x: len(tokenizer.encode(str(x))))
df["a_tokens"] = df["Answers"].apply(lambda x: len(tokenizer.encode(str(x))))

print(df[["q_tokens","a_tokens"]].describe())

config.json:   0%|          | 0.00/896 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

           q_tokens      a_tokens
count  38233.000000  38233.000000
mean     106.640258    114.790783
std       98.283892    274.170000
min        2.000000      2.000000
25%       58.000000     38.000000
50%       87.000000     61.000000
75%      126.000000     93.000000
max     3020.000000   6272.000000


In [ ]:
def full_tokens(row):
    text = f"[INST]\n{row['Questions']}\n[/INST]\nCounselor:\n{row['Answers']}"
    return len(tokenizer.encode(text))

df["total_tokens"] = df.apply(full_tokens, axis=1)

print(df["total_tokens"].describe())
print("Max:", df["total_tokens"].max())

count    38233.000000
mean       230.733319
std        331.793489
min         15.000000
25%        120.000000
50%        162.000000
75%        221.000000
max       6741.000000
Name: total_tokens, dtype: float64
Max: 6741


In [ ]:
print(len(df))

38233


In [ ]:
df = df[df["a_tokens"] > 35]
df = df[df["total_tokens"] < 2048]
print(len(df))


29651


In [ ]:
def format_prompt(row):
    return f"""[INST]
Topic: {row['Topics']}
User: {row['Questions']}
Respond empathetically as a counselor.
[/INST]
Counselor:
{row['Answers']}"""
df["text"] = df.apply(format_prompt, axis=1)
print(df["text"].iloc[10])

[INST]
Topic: উদ্বেগ, সম্পর্ক
User: যখনই আমি আমার বান্ধবীকে ছেড়ে চলে যাই তখনই আমি প্যানিক অ্যাটাক পাই। আমি তাদের নিয়ন্ত্রণ করার জন্য ওষুধ নিচ্ছি, কিন্তু আমি খুব উদ্বিগ্ন হওয়ার পর থেকে আমি তার সাথে যাওয়ার কথা ভাবছি।
Respond empathetically as a counselor.
[/INST]
Counselor:
  আপনি বর্তমানে প্যানিক অ্যাটাকের সম্মুখীন হচ্ছেন শুনে আমি দুঃখিত। আমি আশা করি যে ওষুধটি আপনাকে দেওয়া হয়েছে তা আপনাকে কিছুটা স্বস্তি দিয়েছে। দুর্ভাগ্যবশত, আমি মনে করি না যে আপনার গার্লফ্রেন্ডের সাথে চলাফেরা আপনার উদ্বেগকে কমিয়ে দেবে। যদিও আপনি তাকে ছেড়ে যাওয়ার বিষয়ে খুব উদ্বিগ্ন বোধ করতে পারেন, আপনার শরীর অস্বাস্থ্যকর উপায়ে আপনার জীবনের এই চাপপূর্ণ ঘটনার প্রতিক্রিয়া করছে। আপনি হয়ত এই বিশেষ পরিস্থিতিটি রেন্ডার করতে সক্ষম হবেন কিন্তু একটি অতিরিক্ত চাপের ঘটনা ঘটলে আপনি আরেকটি প্যানিক অ্যাটাকের সম্মুখীন হতে পারেন৷ আপনি কেন প্যানিক অ্যাটাকের সম্মুখীন হচ্ছেন তার কারণ, তারপরে ভবিষ্যতে এমন পরিস্থিতির মোকাবিলা করার পদ্ধতি অনুশীলন করুন। ব্যায়াম এবং যোগব্যায়াম। একজন প্রশিক্ষিত থেরাপিস্ট আপনাকে সঠিক চাপ কমানোর পদ্

**Dataprocessor**

In [ ]:
import unsloth
import torch
import pandas as pd
from datasets import Dataset
from datetime import datetime

from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu

/tmp/ipykernel_23/2746015506.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
class DatasetProcessor:

    def __init__(self, dataframe):
        self.df = dataframe

    def create_prompt(self):

        def format_prompt(row):
            return f"""[INST]
                    Topic: {row['Topics']}
                    User: {row['Questions']}
                    Respond empathetically as a counselor.
                    [/INST]
                    Counselor:
                    {row['Answers']}"""

        self.df["text"] = self.df.apply(format_prompt, axis=1)

    def to_dataset(self):
        from datasets import DatasetDict

        full_dataset = Dataset.from_pandas(self.df[["text"]])

        # Step 1: 95% train, 5% hold-out
        split1 = full_dataset.train_test_split(test_size=0.05, seed=42)
        train_ds = split1["train"]

        # Step 2: split 5% hold-out → 4.5% test + 0.5% validation
        split2 = split1["test"].train_test_split(test_size=0.10, seed=42)
        test_ds       = split2["train"]
        validation_ds = split2["test"]

        print(f"Split sizes → train: {len(train_ds)} | test: {len(test_ds)} | validation: {len(validation_ds)}")

        return DatasetDict({
            "train":      train_ds,
            "test":       test_ds,
            "validation": validation_ds,
        })

In [ ]:
processor = DatasetProcessor(df)
processor.create_prompt()

dataset = processor.to_dataset()

Split sizes → train: 28168 | test: 1334 | validation: 149


In [ ]:
# # Smoke test
# dataset["train"]      = dataset["train"].select(range(min(100, len(dataset["train"]))))
# dataset["test"]       = dataset["test"].select(range(min(10, len(dataset["test"]))))
# dataset["validation"] = dataset["validation"].select(range(min(5, len(dataset["validation"]))))

# print(f"Smoke test → train: {len(dataset['train'])} | test: {len(dataset['test'])} | validation: {len(dataset['validation'])}")

In [ ]:
class TrainingStrategy:

    def load_model(self):
        raise NotImplementedError

In [ ]:
class UnslothStrategy(TrainingStrategy):

    def load_model(self):

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name="unsloth/meta-llama-3.1-8b-instruct-bnb-4bit",
            max_seq_length=2048,
            load_in_4bit=True,
        )

        model = FastLanguageModel.get_peft_model(
            model,
            r=16,
            target_modules=[
                "q_proj","k_proj","v_proj","o_proj",
                "gate_proj","up_proj","down_proj"
            ],
            lora_alpha=16,
            lora_dropout=0,
        )

        return model, tokenizer

In [ ]:
class LLAMAFineTuner:

    def __init__(self, strategy):
        self.strategy = strategy
        self.model, self.tokenizer = strategy.load_model()

    def train(self, dataset):

        training_args = TrainingArguments(
            output_dir="outputs",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,#smoke test 8
            learning_rate=2e-4,
            num_train_epochs=1,
            fp16=True,
            logging_steps=50,#smoke test 50
            save_steps=500,
            # --- VALIDATION LOSS DURING TRAINING ---
            eval_strategy="steps",
            eval_steps=500,#smoke test 500
            per_device_eval_batch_size=1,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            # ----------------------------------------
            # --- AUTOMATIC PUSH CONFIGURATION ---
            push_to_hub=True,
            hub_model_id="kazol196295/llama-3.1-8b-bengali-mental-health-counsellor",
            hub_strategy="every_save",
            # ------------------------------------
            average_tokens_across_devices=False,
            optim="adamw_8bit",
            gradient_checkpointing=True,
        )

        trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            train_dataset=dataset["train"],
            eval_dataset=dataset["validation"],  # dedicated validation split
            dataset_text_field="text",
            max_seq_length=2048,
            args=training_args,
            packing=True,
        )

        trainer.train()

        return trainer

In [ ]:
strategy = UnslothStrategy()

tuner = LLAMAFineTuner(strategy)

trainer = tuner.train(dataset)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/28168 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/149 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 28,168 | Num Epochs = 1 | Total steps = 1,761
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss,Validation Loss
500,0.571641,0.566119
1000,0.544868,0.544218
1500,0.532630,0.531437


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_co